read sample csv, download images from url column into data/images,save update updated csc with local path column 

In [4]:
import os
import time
import pandas as pd
import requests
from tqdm import tqdm
from urllib.parse import urlparse

SAMPLE_CSV = "/Users/mariaworkman/fashion/fashion-neutrality/data/samples/sample_2000_2023_100peryear.csv"
OUT_DIR = "../data/images/sample_2000_2023_100peryear"
OUT_CSV = "../data/samples/sample_2000_2023_100peryear_with_paths.csv"

os.makedirs(OUT_DIR, exist_ok=True)

In [5]:
df = pd.read_csv(SAMPLE_CSV)
df.shape, df.columns
df[["year", "url", "filename"]].head()

,year,url,filename
0,2000,https://assets.vogue.com/photos/55c6514f08298d...,0655677_Rebecca Danenberg - Spring 2000 Ready-...
1,2000,https://assets.vogue.com/photos/55c6514e08298d...,0560590_Miu Miu - Spring 2000 Ready-to-Wear [C...
2,2000,https://assets.vogue.com/photos/55c6516e08298d...,0444360_Halston - Fall 2000 Ready-to-Wear [Col...
3,2000,https://assets.vogue.com/photos/5a3aba11216e3e...,0980151_Helmut Lang - Fall 2000 Ready-to-Wear ...
4,2000,https://assets.vogue.com/photos/55c6514e08298d...,0068211_Loewe - Spring 2000 Ready-to-Wear [Col...


everything looks great. now we create clean consistence filenames for images and download images from URLS with retry logic in case it fails 

In [6]:
def safe_filename_from_row(row, i):
    """
    Prefer dataset filename; otherwise make a stable name from key/year/index.
    """
    if "filename" in row and isinstance(row["filename"], str) and row["filename"].strip():
        # strip weird path separators just in case
        return row["filename"].replace("/", "_")
    if "key" in row:
        return f"{int(row['key']):07d}_y{int(row['year'])}.jpg"
    return f"img_{i:06d}_y{int(row['year'])}.jpg"

def download_with_retries(url, out_path, retries=3, timeout=30, sleep=1.0):
    """
    Downloads url to out_path with a few retries.
    Returns True if saved, False otherwise.
    """
    headers = {"User-Agent": "Mozilla/5.0"}  # helps some CDNs

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, headers=headers, timeout=timeout)
            r.raise_for_status()

            # Basic content-type guard (optional)
            ctype = r.headers.get("Content-Type", "")
            if "image" not in ctype:
                # sometimes CDNs return HTML error pages
                raise ValueError(f"Non-image content-type: {ctype}")

            with open(out_path, "wb") as f:
                f.write(r.content)
            return True

        except Exception as e:
            if attempt == retries:
                return False
            time.sleep(sleep * attempt)  # simple backoff

In [7]:
df = df.copy()
df["local_path"] = None
df["download_ok"] = False

TEST_N = 20
test_df = df.head(TEST_N)

In [8]:
for i, row in tqdm(list(test_df.iterrows()), total=len(test_df)):
    url = row["url"]
    fname = safe_filename_from_row(row, i)

    # keep extension if present in filename, else default .jpg
    if "." not in fname:
        fname += ".jpg"

    out_path = os.path.join(OUT_DIR, fname)

    # skip if already exists
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        df.at[i, "local_path"] = out_path
        df.at[i, "download_ok"] = True
        continue

    ok = download_with_retries(url, out_path, retries=3)
    df.at[i, "local_path"] = out_path if ok else None
    df.at[i, "download_ok"] = ok

100%|██████████| 20/20 [00:10<00:00,  1.92it/s]


In [9]:
df.head(TEST_N)[["year", "download_ok", "local_path"]]
df["download_ok"].head(TEST_N).mean()

np.float64(1.0)

yay, everything downladed....

In [11]:
for i, row in tqdm(list(df.iterrows()), total=len(df)):
    if df.at[i, "download_ok"] is True and isinstance(df.at[i, "local_path"], str):
        continue

    url = row["url"]
    fname = safe_filename_from_row(row, i)

    if "." not in fname:
        fname += ".jpg"

    out_path = os.path.join(OUT_DIR, fname)

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        df.at[i, "local_path"] = out_path
        df.at[i, "download_ok"] = True
        continue

    ok = download_with_retries(url, out_path, retries=3)
    df.at[i, "local_path"] = out_path if ok else None
    df.at[i, "download_ok"] = ok

100%|██████████| 2400/2400 [1:22:45<00:00,  2.07s/it]  
